Bing Chen 

ST 554 Project Part 2 

3/19/2026

# Introduction 

In this project, we developed a PySpark-based data validation class named `SparkDataCheck`. Our class is designed to facilitate data validation and exploration. After defining the class and its associated methods, we will apply it to some real-world dataset here to perform various data quality checks, such as missing value detection, numerical range validation, and etc. 


Now, let's import the class and start a spark session. 

In [1]:
from SparkDataCheck import SparkDataCheck
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


spark = SparkSession.builder \
    .appName("SparkDataCheck") \
    .getOrCreate()



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 15:49:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Next, let's create an instance of the class from a `.csv` file. 

In [2]:
air = SparkDataCheck.from_csv(
    spark,
    "air.csv"
)

air.df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-20 21:00:00|   2.2|       137

26/03/20 15:49:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Using this air quality data, let's see some examples of how to use methods in this class. Recall that in this dataset, missing values are tagged with -200 value, so let's replace them with `None` first. 

In [3]:
air.df = air.df.na.replace(-200, None)

## Check if a numeric variable is within a specified range. 

Next, we might want to check if `PT08.S1(CO)` values specifically are all above 0. 


In [4]:
air.within_range("PT08.S1(CO)", lower=0, upper = None).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|                true|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|                true|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        114

26/03/20 15:49:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Let's see if the values are all below 2000. 

In [5]:
air.within_range("PT08.S1(CO)", lower=None, upper = 1300).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|                true|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        114

26/03/20 15:49:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


How about between 0 to 1500?

In [6]:
air.within_range("PT08.S1(CO)", lower=0, upper = 10).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|               false|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        114

26/03/20 15:49:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


What if we accidentally passed through a string variable? 

In [7]:
air.within_range("Date", lower=0, upper = None).df.show()

Column 'Date' is not numeric. 
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|               false|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|  

26/03/20 15:49:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Nice! It prints out message noting that `Date` is not a numeric column and returns the original data frame. 

What if we forgot to pass on any bounds?

In [8]:
air.within_range("PT08.S1(CO)").df.show()

At least one of 'lower' or 'upper' must be provided. 
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|               false|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       14

26/03/20 15:49:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


So, it prints out that at least one bound need to be supplied. Great! 

## Check if a categorical variable is within specified levels. 

Say we want to classify `PT08.S1(CO)` into 3 levels to allow users to quickly understand if air quality is "Safe" or "Hazardous" without knowing technical sensor specs. Consider the 3 ranks below: 

 - Low: Readings below 970
 - Mid: Readings between 970 and 1,180
 - High: Readings above 1,180
 
 So, let's create a new variable for the transformed `PT08.S1(CO)`. 


In [9]:
air.df = air.df.withColumn(
    "PT08_S1(CO)_cat",
    F.when(F.col("`PT08.S1(CO)`") < 970, "Low")
     .when((F.col("`PT08.S1(CO)`") >= 970) & (F.col("`PT08.S1(CO)`") <= 1180), "Mid")
     .when(F.col("`PT08.S1(CO)`") > 1180, "High")
     .otherwise(None)
)

Let's quickly check the results. 

In [10]:
air.df.select("`PT08.S1(CO)`","PT08_S1(CO)_cat").show(10)

+-----------+---------------+
|PT08.S1(CO)|PT08_S1(CO)_cat|
+-----------+---------------+
|       1360|           High|
|       1292|           High|
|       1402|           High|
|       1376|           High|
|       1272|           High|
|       1197|           High|
|       1185|           High|
|       1136|            Mid|
|       1094|            Mid|
|       1010|            Mid|
+-----------+---------------+
only showing top 10 rows


Looks like it's working correctly. Let's try out our method to see if all values are within the levels. 

In [11]:
air.within_levels("PT08.S1(CO)", levels = ["Low", "Mid", "High"]).df.show()

Column 'PT08.S1(CO)' is not a string column.
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|               fal

26/03/20 15:49:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Oops! Accidentally passed through the numeric column of `PT08.S1(CO)` instead of the categorical column we just created! So, let's do that again. 

In [12]:
air.within_levels("PT08_S1(CO)_cat", levels = ["Low", "Mid", "High"]).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|                     true|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92| 

26/03/20 15:49:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Looks good! Let's try another variable.  We can `Absolute Humidity` into 

- Dry: AH < 0.8  
- Optimal: 0.8 ≤ AH ≤ 1.5  
- Humid: AH > 1.5

In [13]:
air.df = air.df.withColumn(
    "abs_humidity_cat",
    F.when(F.col("AH") < 0.8, "Dry")
     .when((F.col("AH") >= 0.8) & (F.col("AH") <= 1.5), "Optimal")
     .when(F.col("AH") > 1.5, "Humid")
     .otherwise(None)
)

In [14]:
air.within_levels("abs_humidity_cat", levels = ["Optimal", "Humid"]).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|           

26/03/20 15:49:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Why is there so many `false`? Oops! Accidentally missed out a level! 

In [15]:
air.within_levels("abs_humidity_cat", levels = ["Dry", "Optimal", "Humid"]).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|           

26/03/20 15:49:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Now it's looking good. 

## Check Missingness 

When we first look at a dataset, we always want to know how complete is the data, are there any missing values, how are they coded? From the data description, we know that missing values are coded -200 and we already replaced that with `None`. So now, let's check where a value is missing for some variables. 

In [16]:
air.is_missing("`PT08.S1(CO)`").df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|  

26/03/20 15:49:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


In [17]:
air.is_missing("AH").df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|AH_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|  

26/03/20 15:49:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


In [18]:
air.is_missing("NMHC(GT)").df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|AH_is_missing|NMHC(GT)_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-------------------+
|  0|3/10/2004|2026-03-20 18

26/03/20 15:49:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


In [19]:
air.is_missing("Date").df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-------------------+---------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|AH_is_missing|NMHC(GT)_is_missing|Date_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-----------------

26/03/20 15:49:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


Looks good!

## Report the min and max of a given numeric column 

We previously checked if `PT08.S1(CO)` is within a specific range of interest. But what if we have no idea where the values should fall? This method is helpful to tell us this information. 


In [20]:
air.min_max("PT08.S1(CO)").show()

+---------------+---------------+
|PT08.S1(CO)_min|PT08.S1(CO)_max|
+---------------+---------------+
|            647|           2040|
+---------------+---------------+



In [21]:
air.min_max("PT08_S1(CO)_cat")

Column 'PT08_S1(CO)_cat' is not numeric. Please give a numeric column.


We can't use this method on string variables! 

I want to see the min and max of `PT08.S1(CO)` for each levels `Absolute Humidity`. 

In [22]:
air.min_max("PT08.S1(CO)", "abs_humidity_cat").show()

+----------------+---------------+---------------+
|abs_humidity_cat|PT08.S1(CO)_min|PT08.S1(CO)_max|
+----------------+---------------+---------------+
|            NULL|           NULL|           NULL|
|           Humid|            760|           1915|
|             Dry|            647|           1973|
|         Optimal|            704|           2040|
+----------------+---------------+---------------+



We can simply see that min and max for all numeric variables! 

In [23]:
air.min_max().show()

26/03/20 15:49:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/20 15:49:11 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


+-------+-------+----------+----------+---------------+---------------+------------+------------+------------+------------+-----------------+-----------------+-----------+-----------+----------------+----------------+-----------+-----------+----------------+----------------+---------------+---------------+-----+-----+------+------+------+------+
|_c0_min|_c0_max|CO(GT)_min|CO(GT)_max|PT08.S1(CO)_min|PT08.S1(CO)_max|NMHC(GT)_min|NMHC(GT)_max|C6H6(GT)_min|C6H6(GT)_max|PT08.S2(NMHC)_min|PT08.S2(NMHC)_max|NOx(GT)_min|NOx(GT)_max|PT08.S3(NOx)_min|PT08.S3(NOx)_max|NO2(GT)_min|NO2(GT)_max|PT08.S4(NO2)_min|PT08.S4(NO2)_max|PT08.S5(O3)_min|PT08.S5(O3)_max|T_min|T_max|RH_min|RH_max|AH_min|AH_max|
+-------+-------+----------+----------+---------------+---------------+------------+------------+------------+------------+-----------------+-----------------+-----------+-----------+----------------+----------------+-----------+-----------+----------------+----------------+---------------+-------------

In [24]:
air.min_max(col_name = None, group_by = "abs_humidity_cat").show()

26/03/20 15:49:11 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , AH
 Schema: _c0, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/air.csv


+----------------+-------+-------+----------+----------+---------------+---------------+------------+------------+------------+------------+-----------------+-----------------+-----------+-----------+----------------+----------------+-----------+-----------+----------------+----------------+---------------+---------------+-----+-----+------+------+------+------+
|abs_humidity_cat|_c0_min|_c0_max|CO(GT)_min|CO(GT)_max|PT08.S1(CO)_min|PT08.S1(CO)_max|NMHC(GT)_min|NMHC(GT)_max|C6H6(GT)_min|C6H6(GT)_max|PT08.S2(NMHC)_min|PT08.S2(NMHC)_max|NOx(GT)_min|NOx(GT)_max|PT08.S3(NOx)_min|PT08.S3(NOx)_max|NO2(GT)_min|NO2(GT)_max|PT08.S4(NO2)_min|PT08.S4(NO2)_max|PT08.S5(O3)_min|PT08.S5(O3)_max|T_min|T_max|RH_min|RH_max|AH_min|AH_max|
+----------------+-------+-------+----------+----------+---------------+---------------+------------+------------+------------+------------+-----------------+-----------------+-----------+-----------+----------------+----------------+-----------+-----------+------------

## Count the number of each levels for a given string variable

We may be interested in how many observations fall under each `PT08.S1(CO)` category we created.

In [25]:
air.count_level("PT08_S1(CO)_cat").show()

+---------------+-----+
|PT08_S1(CO)_cat|count|
+---------------+-----+
|           NULL|  366|
|           High| 2789|
|            Low| 2819|
|            Mid| 3383|
+---------------+-----+



Most of our data falls in to the Mid category! 

Let's try passing 2 variables. 

In [26]:
air.count_level("PT08_S1(CO)_cat", "abs_humidity_cat").show()

+---------------+----------------+-----+
|PT08_S1(CO)_cat|abs_humidity_cat|count|
+---------------+----------------+-----+
|           NULL|            NULL|  366|
|           High|             Dry|  659|
|           High|           Humid|  470|
|           High|         Optimal| 1660|
|            Low|             Dry| 1060|
|            Low|           Humid|  312|
|            Low|         Optimal| 1447|
|            Mid|             Dry| 1019|
|            Mid|           Humid|  536|
|            Mid|         Optimal| 1828|
+---------------+----------------+-----+



The count for each combination of levels are outputted! 

We can also use this function to count how many missing values we have for a variable. Recall that we created `AH_is_missing` variable. We can convert this column to a string and use this method.  

In [29]:
air.df = air.df.withColumn(
    "AH_is_missing_str",
    F.col("AH_is_missing").cast("string")
)

air.count_level("AH_is_missing_str").show()

+-----------------+-----+
|AH_is_missing_str|count|
+-----------------+-----+
|            false| 8991|
|             true|  366|
+-----------------+-----+



We have 366 missing values! 

If we just pass `AH_is_missing`, our program won't accept it! Because we set up our code to only accept string variables, and this is a Boolean column. 

In [31]:
air.count_level("AH_is_missing")

Column 'AH_is_missing' is a numeric column. Please provide a string column!
